## Sprints Data transform Bronze to Silver

In [0]:
%run ../00-common/01-Environment_Config

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.sprints"
silver_table = f"{catalog_name}.{silver_schema}.sprints"

In [0]:
# To use reader api options 
sprints_df = spark.read.table(bronze_table)

# sprints_df = spark.table(bronze_table) - without any reader options
# Both returns the same results

In [0]:
from pyspark.sql import functions as f

In [0]:
column_rename = {
    "date" : "race_date",
    "grid" : "grid_position",
    "laps" : "completed_laps",
    "number" : "car_number",
    "position" : "finish_position",
    "constructorId" : "constructor_id",
    "driverId" : "driver_id",
    "raceName" : "race_name",
    "positionText" : "finish_position_text"
    }

sprints_renamed_df = sprints_df.drop(f.col("url"))\
                                .withColumnsRenamed(column_rename)

In [0]:
# To Drop duplicates and Null values in a column - distinct(), dropDuplicates() & filter()

sprints_filtered_df = sprints_renamed_df.filter(
    (
        (f.col("driver_id").isNotNull()) & (f.col("constructor_id").isNotNull()) & (f.col("round").isNotNull()) & (f.col("season").isNotNull())
    )
).dropDuplicates(["driver_id","constructor_id","round","season"])

In [0]:
sprints_final_df = sprints_filtered_df.withColumn("race_name", f.initcap("race_name"))

race_date,race_name,round,season,constructor_id,driver_id,grid_position,completed_laps,car_number,points,finish_position,finish_position_text,status,ingestion_timestamp,source_file
2021-07-18,British Grand Prix,10,2021,red_bull,max_verstappen,2,17,33,3.0,1,1,Finished,2026-10-02T20:21:11.902Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2021.json
2021-07-18,British Grand Prix,10,2021,mercedes,hamilton,1,17,44,2.0,2,2,Finished,2026-10-02T20:21:11.902Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2021.json
2021-07-18,British Grand Prix,10,2021,mercedes,bottas,3,17,77,1.0,3,3,Finished,2026-10-02T20:21:11.902Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2021.json
2021-07-18,British Grand Prix,10,2021,ferrari,leclerc,4,17,16,0.0,4,4,Finished,2026-10-02T20:21:11.902Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2021.json
2021-07-18,British Grand Prix,10,2021,mclaren,norris,6,17,4,0.0,5,5,Finished,2026-10-02T20:21:11.902Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2021.json
2021-07-18,British Grand Prix,10,2021,mclaren,ricciardo,7,17,3,0.0,6,6,Finished,2026-10-02T20:21:11.902Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2021.json
2021-07-18,British Grand Prix,10,2021,alpine,alonso,11,17,14,0.0,7,7,Finished,2026-10-02T20:21:11.902Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2021.json
2021-07-18,British Grand Prix,10,2021,aston_martin,vettel,10,17,5,0.0,8,8,Finished,2026-10-02T20:21:11.902Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2021.json
2021-07-18,British Grand Prix,10,2021,williams,russell,8,17,63,0.0,9,9,Finished,2026-10-02T20:21:11.902Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2021.json
2021-07-18,British Grand Prix,10,2021,alpine,ocon,13,17,31,0.0,10,10,Finished,2026-10-02T20:21:11.902Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2021.json


In [0]:
sprints_final_df.write.format('delta').mode('overwrite').saveAsTable(silver_table)